# Jett 2 Holiday — Track 1: ML Demand Forecasting & Telemetry Analysis
**Problem Statement**: APS-02 Real-Time Dynamic Pricing & Demand Forecasting

This notebook demonstrates:
1. **Ingestion of 20,000 telemetry events** from `APS-02.db` (`pricing_events`, `inventory_calendar`, `price_bounds`).
2. **Outlier Defense**: Applying an IQR (Interquartile Range) filter to clip the deliberate ~1% rate outliers.
3. **Feature Engineering**: Computing Recency Decay $\exp(-k_r \cdot \Delta t)$ and Lead-Time Booking Commitment $\exp(-k_c \cdot \text{lead\_time})$.
4. **Linear Regression & Factor Attribution**: Training the model that directly decomposes coefficients into the explainability dashboard.

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from demand_forecaster import DemandForecaster

# Connect to the real APS-02 database
db_path = os.path.join("..", "data", "APS-02.db")
conn = sqlite3.connect(db_path)
print(f"Connected to SQLite: {db_path}")

## 1. Inspect Event Telemetry Distribution

In [ ]:
df_events = pd.read_sql_query("SELECT * FROM pricing_events;", conn)
print(f"Total Events Ingested: {len(df_events):,}")
print("\nEvent Type Breakdown:")
print(df_events['event_type'].value_counts())

## 2. 1% IQR Outlier Defense Filter

In [ ]:
df_events['quoted_price_num'] = pd.to_numeric(df_events['quoted_price'], errors='coerce')
valid_prices = df_events['quoted_price_num'].dropna()

q25, q75 = valid_prices.quantile(0.25), valid_prices.quantile(0.75)
iqr = q75 - q25
lower_bound = q25 - 1.5 * iqr
upper_bound = q75 + 1.5 * iqr

outliers = df_events[(df_events['quoted_price_num'] < lower_bound) | (df_events['quoted_price_num'] > upper_bound)]
print(f"Detected Outlier Events: {len(outliers):,} ({len(outliers)/len(valid_prices)*100:.2f}% of price-bearing events)")
print(f"IQR Clean Range: [{lower_bound:.2f}, {upper_bound:.2f}]")

## 3. Train DemandForecaster Model & Extract Coefficients

In [ ]:
forecaster = DemandForecaster(db_path)
forecaster.train_model()

print("\nModel Feature Weights:")
print(f" - Telemetry Signal Weight: {forecaster.model.coef_[0]:.4f}")
print(f" - Inventory Occupancy Weight: {forecaster.model.coef_[1]:.4f}")
print(f" - Weekend Lift Weight: {forecaster.model.coef_[2]:.4f}")

## 4. 30-Day Demand & Price Horizon Simulation

In [ ]:
# Run inference for sample room type entity
sample_entity = "rmt_ca391f47"
horizon = forecaster.predict_horizon(sample_entity, "2026-10-12", num_days=30)
df_horizon = pd.DataFrame(horizon)

plt.figure(figsize=(12, 5))
plt.plot(df_horizon['date'], df_horizon['current_dynamic_price'], label='Dynamic Published Rate (₹)', color='teal', linewidth=2)
plt.plot(df_horizon['date'], df_horizon['base_price'], label='Baseline Rate', color='gray', linestyle='--')
plt.axhline(y=df_horizon['ceiling_price'].iloc[0], color='red', linestyle=':', label='Ceiling Guardrail')
plt.axhline(y=df_horizon['floor_price'].iloc[0], color='green', linestyle=':', label='Floor Guardrail')
plt.xticks(rotation=45)
plt.title(f'30-Day Dynamic Pricing & Demand Curve for {sample_entity}')
plt.ylabel('Rate (₹)')
plt.legend()
plt.tight_layout()
plt.show()